In [ ]:
import obsidian
print(f'obsidian version: ' + obsidian.__version__)

from obsidian.experiment import AdvExpDesigner

In [ ]:
# Define continuous parameters: key -> (low, high, step)

continuous_params = {
    'temperature': (20, 80, 5),                     # Linear steps of 5 between 20 and 80
    'concentration': (0.1, 1.0, 0.1),               # Linear steps of 0.1 between 0.1 and 1.0
    'pressure': (1, 16, 'geometric'),               # Geometric steps doubling from 1 to 16 (1, 2, 4, 8, 16)
    'time': (10, 1000, 'logarithmic'),              # Logarithmic steps (powers of 10) between 10 and 1000
    'flow_rate': [0.5, 1.0, 2.0, 5.0, 10.0],        # Custom discrete levels, equal biases
    'Reagent Concentration': {
        'levels': [1.0, 2.0, 3.0, 5.0, 10.0],       # Custom levels with biased sampling
        'biases': [0.1, 0.2, 0.4, 0.2, 0.1]         # Higher probability for middle values
}

}

# Define conditional categorical parameters with subparameters and frequencies: key -> {subkey: {'freq': frequency, 'subparams': ([values], [frequencies])}}

conditional_subparameters = {
    'buffer_type': {
        'A': {'freq': 0.4, 'pH': ([6.0, 7.0, 8.0], [0.3, 0.4, 0.3])},
        'B': {'freq': 0.35, 'pH': ([5.0, 6.5], [0.7, 0.3])},
        'C': {'freq': 0.25, 'pH': ([7.5, 8.5], [0.6, 0.4])}
    },
    'catalyst': {
        'X': {'freq': 0.5, 'loading': ([0.1, 0.2, 0.3], [0.2, 0.5, 0.3])},
        'Y': {'freq': 0.3, 'loading': ([0.05, 0.15], [0.6, 0.4])},
        'Z': {'freq': 0.2, 'loading': ([0.25, 0.35], [0.7, 0.3])}
    }
}


# Initialize the designer
designer = AdvExpDesigner(continuous_params, conditional_subparameters)

In [ ]:
# Generate a design with 100 samples, optimizing categorical assignments
design = designer.generate_design(seed=123, n_samples=100, optimize_categories=True)
design

In [ ]:
# Evaluate the design quality metrics
metrics = designer.evaluate_design(design)
print("Design quality metrics:")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Plot histograms of all parameters and subparameters
designer.plot_histograms(design)

In [ ]:
# Plot PCA colored by 'buffer_type'
designer.plot_pca(design, hue='buffer_type')

# Plot UMAP colored by 'catalyst'
designer.plot_umap(design, hue='catalyst')

In [ ]:
# Optimize design over 30 trials with 100 samples each
best_design, metrics_df = designer.optimize_design(n_trials=30, n_samples=100)

print("\nBest design metrics after optimization:")
print(metrics_df.sort_values('score', ascending=False).head(1))


In [ ]:
# Plot quality evolution over trials
designer.plot_quality_evolution(metrics_df)

In [ ]:
# Plot correlation matrix of the design
designer.plot_correlation(best_design)

In [ ]:
# Extend the best design by 20 new samples over 10 trials
extended_design, extension_summary = designer.extend_design(best_design, n=20, n_trials=10)

print("\nExtension summary:")
print(extension_summary)

# Plot the extended design
designer.plot_histograms(extended_design)

In [ ]:
# Compare empirical vs expected frequencies for categorical variables
designer.compare_frequencies(extended_design)

## Save and restore the designer

Both designers are restartable. A configured designer can be serialized to a JSON-safe
dictionary with `save_state()` and rebuilt later with `load_state(...)`:

- **`ExpDesigner`** (the basic designer) round-trips its parameter space and seed.
- **`AdvExpDesigner`** additionally preserves the continuous params, conditional
  subparameters, subparameter mapping, and any attached design.

This lets you persist a designer configuration and resume where you left off — the same
round-trip `Campaign.save_state()` performs when it embeds its designer. The example below
uses `AdvExpDesigner`; the basic designer works identically via `ExpDesigner.load_state(...)`.

In [ ]:
import json
import pandas as pd

# Serialize the configured designer to a JSON-safe dict
state = designer.save_state()
print("Saved keys:", list(state))

# A real workflow would persist this to disk, e.g.:
#   with open('designer_state.json', 'w') as f:
#       json.dump(state, f)
# Here we round-trip through a JSON string to confirm the payload is JSON-safe.
restored = AdvExpDesigner.load_state(json.loads(json.dumps(state)))

# The configuration is faithfully reconstructed
print("Continuous params match:  ", list(restored.continuous_params) == list(designer.continuous_params))
print("Conditional params match: ", list(restored.conditional_subparameters) == list(designer.conditional_subparameters))

# A deterministic design (optimize_categories=False) reproduces the same values after reload.
# (check_dtype=False tolerates a benign int->float normalization of integer-valued levels.)
orig = designer.generate_design(seed=123, n_samples=100, optimize_categories=False)
repro = restored.generate_design(seed=123, n_samples=100, optimize_categories=False)
try:
    pd.testing.assert_frame_equal(orig, repro, check_dtype=False)
    print("Design reproduced exactly:", True)
except AssertionError:
    print("Design reproduced exactly:", False)